In [1]:
import os
os.chdir('application')

In [2]:
import uuid

from typing import List
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import redis

from modules.redis import RedisConversationMemory
# from modules.retriever_wrappper import SanitizingRetriever
from modules.history_utils import build_chat_history_messages

from config import (REDIS_URL, SESSION_TTL_SECONDS, MAX_HISTORY_MESSAGES, 
                    gemini_chat_model, api_key_name, system_prompt
)
from vectorstore.milvus_vectorstore import get_vectorstore

import logging

# LangChain imports (use provider wrapper)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# No contextualization

In [4]:
# ----- CONFIG -----
r = redis.Redis.from_url(REDIS_URL, decode_responses=True)
app = FastAPI()

# Your already-created vectorstore (LangChain Milvus wrapper)
vectorstore = get_vectorstore()


In [6]:
mem = RedisConversationMemory(session_id='001', max_messages=MAX_HISTORY_MESSAGES)

In [7]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

In [8]:
llm = ChatGoogleGenerativeAI(model=gemini_chat_model, google_api_key=os.environ.get(api_key_name))

In [3]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [10]:
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=None,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": qa_prompt},
)


ValidationError: 1 validation error for StuffDocumentsChain
  Value error, document_variable_name context was not found in llm_chain input_variables: ['input'] [type=value_error, input_value={'llm_chain': LLMChain(ve...None, 'callbacks': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

# Contextualization